# Neocortex (Jorstad 2023): which supercluster did each downsampled cell come from, and the values for the master summary

**Objective.** Ray's ETL needs `master_dataset_summary` to hold one row for each of the five superclusters, each with its own `dataset_version_id` and its own values. This notebook makes those rows, step by step, from the files the run already has. Nothing is filtered.

The notebook is written in the way Knuth (1984, *Literate Programming*, The Computer Journal 27:97-111) asks for and in the notebook style of Rule et al. (2019, *Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks*, PLoS Comput Biol 15:e1007007): a short piece of code, then a plain explanation of what it did and why.

**What it uses.**
1. The downsampled file used for the NS-Forest run (73,144 cells, at most 500 cells for each of the 153 clusters).
2. The five supercluster files, in the versions of 2023-10-20, read from CellxGene (only the cell annotations are read, never the gene values).
3. The results folder of the run: the NS-Forest results and the silhouette summary of each cluster.
4. The reference file that says which supercluster each cluster belongs to.

In [1]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd

PROJECTS = Path('/Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects')
H5AD     = PROJECTS / '2026-Oct-22-neocortex-run/adata_sampled_AIBS_human_neocortex_plainstrings.h5ad'
RESULTS  = next((PROJECTS / 'nlm-ckn/data/prod/neocortex/sc-nsforest-qc-nf/results').glob('neocortex-Jorstad-Science-2023-*'))
REFERENCE = next((PROJECTS / 'neocortex_for_review').glob('cluster_cid_mapping_neocortex_*.csv'))
CACHE    = Path.home() / '.cache' / 'cellxgene-harvester-notebooks'     # the cell annotations read from CellxGene are kept here
OUT      = PROJECTS / 'neocortex_for_review' / 'notebook_out'
CACHE.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)

# the five superclusters, in the order of params/neocortex_silhouette_datasets.csv: the versions of 2023-10-20
SUPERCLUSTERS = {
    'IT':  '0476ef54-aefe-4754-b0e9-d9fcd75adff4',
    'MGE': '8b09695a-1426-4867-961e-c40a1fbcc2da',
    'CGE': '3280113b-7148-4a3e-98d4-015f443aab8a',
    'NN':  'f72b8175-8316-4343-86bb-d5171d5ef079',
    'DL':  'c4959ded-83dc-4442-aac7-9a59bdb47801',
}
print(RESULTS.name); print(REFERENCE.name)

neocortex-Jorstad-Science-2023-f52998989531b0e14df842aeeb5df4096ce08e01bdd0bae97a049665543b237c
cluster_cid_mapping_neocortex_Jorstad_Science_2023_CrossArea_cluster_X_UMAP_f52998989531b0e14df842aeeb5df4096ce08e01bdd0bae97a049665543b237c.csv


**Explanation.** These are the places the notebook reads from and writes to. The versions are those of 2023-10-20, the ones the run read. A newer version of each file exists in CellxGene, and the harvester returns those, so the ids are written out here and not looked up.

In [2]:
import requests
rows = []
for name, vid in SUPERCLUSTERS.items():
    j = requests.get(f'https://api.cellxgene.cziscience.com/curation/v1/dataset_versions/{vid}', timeout=30).json()
    rows.append({'supercluster': name, 'dataset_version_id': vid, 'dataset_id': j['dataset_id'],
                 'title': j['title'], 'cells': j['cell_count']})
versions = pd.DataFrame(rows)
versions

  supercluster                    dataset_version_id                            dataset_id                                                 title   cells
0           IT  0476ef54-aefe-4754-b0e9-d9fcd75adff4  c3aa4f95-7a18-4a7d-8dd8-ca324d714363        Supercluster: IT-projecting excitatory neurons  638941
1          MGE  8b09695a-1426-4867-961e-c40a1fbcc2da  9c63201d-bfd9-41a8-bbbc-18d947556f3d                Supercluster: MGE-derived interneurons  185477
2          CGE  3280113b-7148-4a3e-98d4-015f443aab8a  e4ddac12-f48f-4455-8e8d-c2a48a683437                Supercluster: CGE-derived interneurons  129495
3           NN  f72b8175-8316-4343-86bb-d5171d5ef079  5346f9c6-755e-4336-94cc-38706ec00c2f                      Supercluster: Non-neuronal cells  108940
4           DL  c4959ded-83dc-4442-aac7-9a59bdb47801  d01c9dff-abd1-4825-bf30-2eb2ba74597e  Supercluster: Deep layer (non-IT) excitatory neurons   92969

**Explanation.** CellxGene is asked for the title, the dataset id and the number of cells of each of the five versions. The cells add up to 1,155,822, which is the number the silhouette step counted.

In [3]:
import anndata as ad
adata = ad.read_h5ad(H5AD, backed='r')        # backed: only the annotations are read, the gene values stay on disk
obs = adata.obs[['CrossArea_subclass', 'CrossArea_cluster', 'donor_id', 'tissue_ontology_term_id',
                 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'disease_ontology_term_id',
                 'sex_ontology_term_id', 'development_stage_ontology_term_id',
                 'cell_type', 'development_stage', 'is_primary_data']].copy()
obs = obs.astype({c: 'object' for c in obs.columns if str(obs[c].dtype) == 'category'})   # plain values, so that a count never lists a value that no cell has
print(adata.shape, 'cells x genes;', obs.CrossArea_cluster.nunique(), 'clusters')

(73144, 28189) cells x genes; 153 clusters


**Explanation.** The downsampled file is opened without loading the gene values. The columns that are stored as categories are made plain values, so that a count never lists a term that no cell of the group has. Each cell keeps the cluster it was given by the authors (`CrossArea_cluster`) and the CellxGene annotations it came with.

In [4]:
ref = pd.read_csv(REFERENCE, dtype=str)
assert ref.cluster_name.is_unique and len(ref) == 153
cluster_version = ref.set_index('cluster_name').dataset_version_id       # cluster -> supercluster version
version_name = {v: k for k, v in SUPERCLUSTERS.items()}

obs['supercluster'] = obs.CrossArea_cluster.map(cluster_version).map(version_name)
assert obs.supercluster.notna().all()                                      # every cell falls in a known supercluster
down = (obs.groupby('supercluster')
          .agg(cells=('CrossArea_cluster', 'size'), clusters=('CrossArea_cluster', 'nunique'),
               most_cells_in_a_cluster=('CrossArea_cluster', lambda s: s.value_counts().max()))
          .reindex(list(SUPERCLUSTERS)))
down.loc['all'] = [down.cells.sum(), down.clusters.sum(), down.most_cells_in_a_cluster.max()]
down

              cells  clusters  most_cells_in_a_cluster
supercluster                                          
IT            11000        22                      500
MGE           23595        48                      500
CGE           23552        49                      500
NN             6277        15                      500
DL             8720        19                      500
all           73144       153                      500

**Explanation.** A cluster belongs to one supercluster only, so a cell's cluster tells where it came from. The reference file gives the supercluster of each of the 153 clusters; each downsampled cell is given the supercluster of its cluster. All 73,144 cells find a supercluster, the clusters add up to 153, and no cluster has more than the 500 cells of the downsampling.

In [5]:
res = pd.read_csv(next(RESULTS.glob('results_ensg_*.csv')))
check = (obs.groupby('CrossArea_cluster').size().rename('cells_in_file').to_frame()
           .join(res.set_index('clusterName')[['sampled_clusterSize', 'clusterSize']]))
print('cells counted here equal sampled_clusterSize in the results:', (check.cells_in_file == check.sampled_clusterSize).all())
print('no cluster above 500 cells:', (check.cells_in_file <= 500).all(),
      '| a cluster with fewer than 500 has all its cells:', ((check.cells_in_file == check.clusterSize) | (check.cells_in_file == 500)).all())

cells counted here equal sampled_clusterSize in the results: True
no cluster above 500 cells: True | a cluster with fewer than 500 has all its cells: True


**Explanation.** Two checks against the NS-Forest results. The count of cells of each cluster in the downsampled file equals `sampled_clusterSize` in the results. And a cluster has 500 cells, or all the cells it has if it has fewer than 500.

In [6]:
import h5py, fsspec

OBS_COLUMNS = ['CrossArea_cluster', 'donor_id', 'tissue_ontology_term_id', 'tissue', 'assay_ontology_term_id', 'assay',
               'cell_type_ontology_term_id', 'cell_type', 'disease_ontology_term_id', 'disease',
               'development_stage_ontology_term_id', 'development_stage', 'sex_ontology_term_id', 'sex', 'is_primary_data']

def read_obs_remote(version_id):
    """Read the cell annotations of one CellxGene h5ad file over the network, from the cache when it is there."""
    cached = CACHE / f'obs_{version_id[:8]}.pkl'
    if cached.exists():
        return pd.read_pickle(cached)
    url = f'https://datasets.cellxgene.cziscience.com/{version_id}.h5ad'
    f = h5py.File(fsspec.open(url, 'rb', block_size=32 * 1024 * 1024).open(), 'r')
    columns = {}
    for name in OBS_COLUMNS:
        if name not in f['obs']:
            continue
        g = f['obs'][name]
        if isinstance(g, h5py.Group) and 'codes' in g:                    # a category: the codes point into the list of values
            values = np.array([x.decode() if isinstance(x, bytes) else x for x in g['categories'][:]], dtype=object)
            codes = g['codes'][:]
            columns[name] = pd.Series(np.where(codes >= 0, values[np.clip(codes, 0, None)], None), dtype=object)
        else:
            columns[name] = pd.Series([x.decode() if isinstance(x, bytes) else x for x in g[:]], dtype=object)
    frame = pd.DataFrame(columns)
    frame.to_pickle(cached)
    return frame

full = {name: read_obs_remote(vid) for name, vid in SUPERCLUSTERS.items()}
pd.DataFrame({'cells': {n: len(o) for n, o in full.items()},
              'clusters': {n: o.CrossArea_cluster.nunique() for n, o in full.items()}})

      cells  clusters
IT   638941        22
MGE  185477        48
CGE  129495        49
NN   108940        15
DL    92969        19

**Explanation.** For each of the five supercluster files, only the cell annotations are read from CellxGene, over the network (the file is opened remotely and only the annotation columns are fetched, so nothing large is stored). They are kept in a cache folder so that the next run is fast. These are all the cells of each file, not the downsampled ones.

In [7]:
members = {n: set(o.CrossArea_cluster.dropna().unique()) for n, o in full.items()}
disjoint = sum(len(s) for s in members.values()) == len(set().union(*members.values()))
same_as_reference = all(members[n] == set(ref[ref.dataset_version_id == v].cluster_name) for n, v in SUPERCLUSTERS.items())
print('no cluster is in two files:', disjoint, '| the 153 clusters in the files:', len(set().union(*members.values())),
      '| the files agree with the reference:', same_as_reference)
print('cells in the five files:', sum(len(o) for o in full.values()))

no cluster is in two files: True | the 153 clusters in the files: 153 | the files agree with the reference: True
cells in the five files: 1155822


**Explanation.** This is the independent check of the assignment made above from the reference. The clusters found in the five real files are separate (no cluster appears in two files), they are the 153 clusters, and they match the reference supercluster by supercluster. So the cells of the downsampled file can be given to their supercluster with no doubt.

In [8]:
# the cells counted by the harvester and the master summary are described the same way: the term ids with their cell counts
STAGE_AGE = lambda label: (int(m.group(1)) if (m := re.match(r'(\d+)-year-old', str(label))) else None)

def count_terms(frame, id_col, label_col=None):
    counts = frame.groupby(id_col).size()
    ordered = sorted(counts.items(), key=lambda pair: (-pair[1], pair[0]))      # most cells first, then by id
    if label_col is None:
        return '; '.join(f'{i}: {n}' for i, n in ordered), ' | '.join(sorted(counts.index))
    label = frame.dropna(subset=[label_col]).groupby(id_col)[label_col].agg(lambda s: s.value_counts().index[0])
    return '; '.join(f'{label.get(i, i)}: {n}' for i, n in ordered), ' | '.join(sorted(counts.index))

def describe(frame):
    """The cell count, the donors and the six facets of a set of cells (no filter)."""
    out = {'total_cell_count': len(frame), 'donor_id_count': frame.donor_id.nunique()}
    for facet, id_col, label in [('tissue', 'tissue_ontology_term_id', None), ('assay', 'assay_ontology_term_id', None),
                                 ('cell_type', 'cell_type_ontology_term_id', None), ('disease', 'disease_ontology_term_id', None),
                                 ('sex', 'sex_ontology_term_id', None),
                                 ('development_stage', 'development_stage_ontology_term_id', 'development_stage')]:
        summary, ids = count_terms(frame, id_col, label)
        out[f'{facet}_ontology_term_id'] = ids
        out['development_stage_summary' if facet == 'development_stage' else f'{facet}_ontology_summary'] = summary
    return out
print(describe(full['NN'])['cell_type_ontology_summary'])

CL:0000128: 51816; CL:0002605: 27563; CL:0002453: 15670; CL:0000129: 12217; CL:1001602: 909; CL:4023051: 765


**Explanation.** One function describes any set of cells the way the harvester does: how many cells and donors there are, and for tissue, assay, cell type, disease, sex and developmental stage, each term with the number of cells that have it, the largest first. No cell is removed, because the run used every cell (`unfiltered`).

In [9]:
# Check of the method against the harvester: with its filters (neocortex, normal, adult from 15 years)
# the same function must give the numbers of the harvester file of the current version of the non-neuronal cells.
uberon = set()
def collect(x):
    if isinstance(x, dict): [collect(v) for v in x.values()]
    elif isinstance(x, list): [collect(v) for v in x]
    elif isinstance(x, str) and x.startswith('UBERON:'): uberon.add(x)
collect(json.load(open(PROJECTS / 'nlm-ckn/data/prod/neocortex/cellxgene-harvester/uberon_neocortex.json')))

current_NN = read_obs_remote('8b7fdf9f-1fa3-4808-8fcc-a5a5122069ec')          # the current version of the non-neuronal cells
keep = ((current_NN.disease_ontology_term_id == 'PATO:0000461') & current_NN.tissue_ontology_term_id.isin(uberon)
        & (current_NN.development_stage.map(STAGE_AGE) >= 15))
filtered = describe(current_NN[keep])
print('filtered cells:', filtered['total_cell_count'], '(harvester file: 50300) | donors:', filtered['donor_id_count'], '(harvester file: 5)')
print(filtered['sex_ontology_summary'], '(harvester file: PATO:0000384: 46839; PATO:0000383: 3461)')

filtered cells: 50300 (harvester file: 50300) | donors: 5 (harvester file: 5)
PATO:0000384: 46839; PATO:0000383: 3461 (harvester file: PATO:0000384: 46839; PATO:0000383: 3461)


**Explanation.** Before trusting the function, it is run the way the harvester runs: only the neocortex tissues, normal disease and adults from 15 years. On the current version of the non-neuronal cells it gives the cell count, the donor count and the sex counts that the harvester file holds. (Run on all five current versions it matched the cell counts, donor counts and term ids and counts of the harvester file; one version is used here to keep the notebook fast. Two things in the harvester file are not reproduced: the long list of zero-count stages and one row with thousands commas.) The same function without the filter is used below.

In [10]:
by_supercluster = {}
for name, vid in SUPERCLUSTERS.items():
    by_supercluster[name] = describe(full[name])
values = pd.DataFrame(by_supercluster).T
values[['total_cell_count', 'donor_id_count', 'tissue_ontology_term_id', 'cell_type_ontology_term_id']]

    total_cell_count donor_id_count                                                                                                                tissue_ontology_term_id                                                   cell_type_ontology_term_id
IT            638941              6  UBERON:0001384 | UBERON:0002436 | UBERON:0002686 | UBERON:0002771 | UBERON:0008933 | UBERON:0009834 | UBERON:0009835 | UBERON:0034751                                                                   CL:4023040
MGE           185477              6  UBERON:0001384 | UBERON:0002436 | UBERON:0002686 | UBERON:0002771 | UBERON:0008933 | UBERON:0009834 | UBERON:0009835 | UBERON:0034751                                         CL:4023017 | CL:4023018 | CL:4023036
CGE           129495              6  UBERON:0001384 | UBERON:0002436 | UBERON:0002686 | UBERON:0002771 | UBERON:0008933 | UBERON:0009834 | UBERON:0009835 | UBERON:0034751                            CL:4023011 | CL:4023015 | CL:4023016 | CL:4023070
NN      

**Explanation.** The description of all the cells of each of the five supercluster files, the 2023-10-20 versions. These are the cells the silhouette step used. The cell counts equal the counts of CellxGene in the table above.

In [11]:
# the same description for the downsampled cells of each supercluster: the cells the NS-Forest step used
by_supercluster_downsampled = pd.DataFrame({name: describe(obs[obs.supercluster == name]) for name in SUPERCLUSTERS}).T
by_supercluster_downsampled[['total_cell_count', 'donor_id_count', 'cell_type_ontology_term_id']]

    total_cell_count donor_id_count                                                   cell_type_ontology_term_id
IT             11000              6                                                                   CL:4023040
MGE            23595              6                                         CL:4023017 | CL:4023018 | CL:4023036
CGE            23552              6                            CL:4023011 | CL:4023015 | CL:4023016 | CL:4023070
NN              6277              6  CL:0000128 | CL:0000129 | CL:0002453 | CL:0002605 | CL:1001602 | CL:4023051
DL              8720              6                            CL:4023012 | CL:4023013 | CL:4023038 | CL:4023041

**Explanation.** The same, for the downsampled cells. These are the 73,144 cells the NS-Forest markers were made from, divided among the superclusters with the assignment checked above. The master summary is about the datasets that were analysed, so its rows use the full files; this table is kept to show the downsampled counts and to compare.

In [12]:
clusters = pd.read_csv(next(RESULTS.glob('scsilhouette_cluster_summary_*.csv')))       # silhouette of each cluster, computed within its own supercluster
clusters['supercluster'] = clusters.CrossArea_cluster.map(cluster_version).map(version_name)
res['supercluster'] = res.clusterName.map(cluster_version).map(version_name)

def silhouette_and_fscore(c, r):
    """The numbers of scsilhouette compute-summary-stats (version 1.0) for a set of clusters."""
    m = c.median_silhouette
    return {'n_clusters': len(c), 'median_silhouette': m.median(), 'mean_silhouette': m.mean(), 'std_silhouette': m.std(),
            'filtered_cell_count': int(c['count'].sum()),
            'high_quality_clusters': int((m >= 0.5).sum()), 'medium_quality_clusters': int(((m >= 0.25) & (m < 0.5)).sum()),
            'low_quality_clusters': int((m < 0.25).sum()), 'median_fscore': r.f_score.median(), 'mean_fscore': r.f_score.mean()}

whole = silhouette_and_fscore(clusters, res)
published = {'n_clusters': 153, 'median_silhouette': 0.3680647675909828, 'mean_silhouette': 0.3275719526331676,
             'std_silhouette': 0.34348724335947883, 'filtered_cell_count': 1155822, 'high_quality_clusters': 55,
             'medium_quality_clusters': 40, 'low_quality_clusters': 58, 'median_fscore': 0.565749235, 'mean_fscore': 0.5711539252026144}
print('the whole run gives the numbers already published in the master summary:',
      all(np.isclose(whole[k], published[k], rtol=0, atol=1e-9) for k in published))
numbers = pd.DataFrame({n: silhouette_and_fscore(clusters[clusters.supercluster == n], res[res.supercluster == n]) for n in SUPERCLUSTERS}).T
numbers

the whole run gives the numbers already published in the master summary: True


     n_clusters  median_silhouette  mean_silhouette  std_silhouette  filtered_cell_count  high_quality_clusters  medium_quality_clusters  low_quality_clusters  median_fscore  mean_fscore
IT         22.0           0.438035         0.425583        0.222738             638941.0                    7.0                      9.0                   6.0       0.629352     0.614210
MGE        48.0           0.200189         0.226463        0.349472             185477.0                   13.0                     10.0                  25.0       0.491578     0.507800
CGE        49.0           0.326229         0.250119        0.361210             129495.0                   13.0                     14.0                  22.0       0.535308     0.545265
NN         15.0           0.631934         0.551382        0.247532             108940.0                   11.0                      3.0                   1.0       0.751263     0.700523
DL         19.0           0.657546         0.492573        0.3132

**Explanation.** The silhouette scores and the F-scores are not new numbers: the run already has them for each cluster (the silhouette was computed within each supercluster file, as Renee's instructions say). The numbers of a supercluster are the same summary the scsilhouette tool makes for the whole run (median of the cluster medians, the count of high, medium and low clusters, the median and mean F-score), taken over the clusters of that supercluster only. As a check, the formulas applied to all 153 clusters give exactly the numbers already published in the master summary. The tool itself (`scsilhouette:v1.0`, `compute-summary-stats`) was also run on each supercluster and gave the same values as the table.

In [13]:
published_row = pd.read_csv(next(RESULTS.glob('master_dataset_summary_*.csv')), dtype=str, keep_default_na=False)
base = published_row.iloc[0].to_dict()
rows = []
for name, vid in SUPERCLUSTERS.items():
    row = dict(base)
    row['dataset_version_id'] = vid
    row['h5ad_url'] = f'https://datasets.cellxgene.cziscience.com/{vid}.h5ad'
    row['dataset'] = f'neocortex_Jorstad_Science_2023_CrossArea_cluster_X_UMAP_{vid[-6:]}'      # the name the tool gives: the last 6 characters of the id
    for key, value in by_supercluster[name].items():
        if key.endswith('_summary'):
            value = value.replace('; ', ' | ') if value else 'NA'                  # the format the master summary uses
        if key in row:
            row[key] = value
    for key, value in numbers.loc[name].items():
        row[key] = value
    rows.append(row)
master = pd.DataFrame(rows)[list(published_row.columns)]
master.to_csv(OUT / next(RESULTS.glob('master_dataset_summary_*.csv')).name, index=False)
master[['dataset_version_id', 'n_clusters', 'filtered_cell_count', 'median_silhouette', 'median_fscore']]

                     dataset_version_id  n_clusters  filtered_cell_count  median_silhouette  median_fscore
0  0476ef54-aefe-4754-b0e9-d9fcd75adff4        22.0             638941.0           0.438035       0.629352
1  8b09695a-1426-4867-961e-c40a1fbcc2da        48.0             185477.0           0.200189       0.491578
2  3280113b-7148-4a3e-98d4-015f443aab8a        49.0             129495.0           0.326229       0.535308
3  f72b8175-8316-4343-86bb-d5171d5ef079        15.0             108940.0           0.631934       0.751263
4  c4959ded-83dc-4442-aac7-9a59bdb47801        19.0              92969.0           0.657546       0.632267

**Explanation.** The five rows. Each starts from the row the run wrote, then takes its own `dataset_version_id` and `h5ad_url` (the CellxGene address), the description of its cells, and its own silhouette and F-score numbers. The columns and their order are those of the published file. The file is written to `neocortex_for_review/notebook_out` and nothing in `nlm-ckn` is changed.

**What this notebook did not do.** It did not run the cellxgene-harvester. The harvester returns the current versions of the datasets, not those of 2023-10-20 that the run read, and it cannot be run without filters. The description of the cells is made with the same counting rules and was checked against the harvester output, as shown above.